# 03 · Conexão robusta e a primeira aplicação web

**Pré-requisito:** notebooks 00 a 02.

Este notebook é um **passo a passo completo**: todas as células já estão preenchidas. Rode uma por vez, leia o que cada uma faz e observe o resultado.

**Nesta etapa:**
- uma conexão que **explica os erros** em português;
- o **Streamlit**, que transforma um script Python num site;
- a **Secretaria online**: um app que cadastra e lista alunos do seu MongoDB, rodando no navegador.

In [ ]:
#%pip install -q pymongo python-dotenv pillow numpy matplotlib requests streamlit pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Parte 1 · Por que o endereço do banco fica fora do código

Hoje o seu banco está em `mongodb://localhost:27017`, sem senha. Em produção, o endereço muda (outro servidor, a nuvem) e passa a ter **usuário e senha**. Se ele estiver escrito no código, cada mudança obriga a editar todos os arquivos, e a senha acaba indo parar num print ou num repositório público.

Por isso o endereço fica **fora do código**:

| Onde o código roda | Onde fica o endereço | Como ler no Python |
|---|---|---|
| Notebook no VS Code | arquivo `.env` | `load_dotenv()` + `os.environ["MONGO_URI"]` |
| App Streamlit | `.streamlit/secrets.toml` | `st.secrets["MONGO_URI"]` |

---
## Parte 2 · Uma conexão que explica os erros

Quando a conexão falha, o PyMongo mostra mensagens longas, em inglês. A função `conectar` abaixo tenta um `ping` e, se der errado, traduz o problema:

| Erro do PyMongo | O que significa |
|---|---|
| `ServerSelectionTimeoutError` | ninguém respondeu nesse endereço: o MongoDB está desligado ou a porta está errada |
| `ConfigurationError` | o endereço está mal escrito |

O `serverSelectionTimeoutMS=3000` faz o erro aparecer em 3 segundos, em vez de 30.

In [2]:
import os
from pymongo import MongoClient
from pymongo.errors import ConfigurationError, ServerSelectionTimeoutError

def conectar(uri):
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=3000)
        client.admin.command("ping")
        print("Conectado")
        return client
    except ServerSelectionTimeoutError:
        print("ERRO: Nenhum MongoDB respondeu nesse endereço/porta. O serviço está rodando? (Get-Service MongoDB)")
    except ConfigurationError as e:
        print("ERRO: Endereço mal escrito:", e)
    return None

Com o endereço certo, lido do `.env`:

In [3]:
from dotenv import load_dotenv

load_dotenv()
uri = os.environ["MONGO_URI"]
client = conectar(uri)

Conectado


Agora dois erros **de propósito**, para ver as mensagens:
- a porta `27018`, onde não há MongoDB (espera 3 segundos);
- `mongdb://`, com o “o” faltando.

In [4]:
conectar("mongodb://localhost:27018")   # porta sem MongoDB
conectar("mongdb://localhost:27017")    # endereço mal escrito

ERRO: Nenhum MongoDB respondeu nesse endereço/porta. O serviço está rodando? (Get-Service MongoDB)
ERRO: Endereço mal escrito: Invalid URI scheme: URI must begin with 'mongodb://' or 'mongodb+srv://'


Com a conexão certa, recriamos a coleção `escola.alunos` que o app vai usar:

In [5]:
ALUNOS = [
    {"nome": "Ana",     "idade": 22, "curso": "Computação"},
    {"nome": "Bruno",   "idade": 19, "curso": "Medicina"},
    {"nome": "Carla",   "idade": 24, "curso": "Direito"},
    {"nome": "Matheus", "idade": 21, "curso": "Computação", "disciplinas": ["Python"]},
    {"nome": "Mariana", "idade": 26, "curso": "Engenharia"},
    {"nome": "Pedro",   "idade": 20, "curso": "Medicina"},
    {"nome": "Juliana", "idade": 23, "curso": "Ciência de Dados"},
    {"nome": "Lucas",   "idade": 18, "curso": "Direito"},
]

alunos = client["escola"]["alunos"]
alunos.delete_many({})
alunos.insert_many(ALUNOS)
print(alunos.count_documents({}), "alunos em escola.alunos")

8 alunos em escola.alunos


## Parte 3 · Streamlit: um script vira um site

O Streamlit transforma um arquivo `.py` numa página web, sem HTML nem JavaScript. Cada comando `st.` desenha uma parte da página:

| Comando | Para que serve |
|---|---|
| `st.title`, `st.subheader` | títulos |
| `st.text_input`, `st.number_input`, `st.selectbox` | campos de entrada |
| `st.form` + `st.form_submit_button` | agrupa campos e envia ao clicar |
| `st.dataframe`, `st.metric` | mostrar tabelas e números |
| `st.secrets` | ler o endereço do banco sem escrevê-lo no código |
| `@st.cache_resource` | abrir a conexão uma vez e reaproveitar |

> **O segredo do Streamlit:** a cada clique, o script roda **inteiro** de novo, de cima para baixo. Por isso a conexão com o banco fica numa função com `@st.cache_resource`; sem ele, cada clique abriria uma conexão nova com o banco.

O app fica numa pasta própria, `secretaria`. A célula abaixo cria a pasta e, dentro dela, o arquivo de segredos do Streamlit (`.streamlit/secrets.toml`) a partir do seu `.env`.

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
os.makedirs("secretaria/.streamlit", exist_ok=True)
with open("secretaria/.streamlit/secrets.toml", "w", encoding="utf-8") as f:
    f.write(f'MONGO_URI = "{os.environ["MONGO_URI"]}"\n')
print("Criados: secretaria/  e  secretaria/.streamlit/secrets.toml")

Criados: secretaria/  e  secretaria/.streamlit/secrets.toml


A próxima célula **grava o app** no arquivo `secretaria/app.py` (é isso que o `%%writefile` faz: a célula não roda, vira um arquivo). O app tem três blocos:

1. **Cadastro:** um formulário com nome, idade e curso; ao clicar em *Salvar*, faz `insert_one`.
2. **Filtro:** um `selectbox` com “Todos” + os cursos existentes (`distinct`), que monta o filtro do `find`.
3. **Indicadores e tabela:** total de alunos e idade média do filtro, e a tabela com os alunos.

In [7]:
%%writefile secretaria/app.py
import pandas as pd
import streamlit as st
from pymongo import MongoClient

st.set_page_config(page_title="Secretaria online")

@st.cache_resource
def conectar():
    client = MongoClient(st.secrets["MONGO_URI"])
    return client["escola"]["alunos"]

alunos = conectar()
st.title("Secretaria online")

# 1. Cadastro
with st.form("cadastro", clear_on_submit=True):
    st.subheader("Cadastrar aluno")
    nome = st.text_input("Nome")
    idade = st.number_input("Idade", min_value=14, max_value=99, value=20, step=1)
    curso = st.text_input("Curso")
    if st.form_submit_button("Salvar"):
        if nome and curso:
            alunos.insert_one({"nome": nome, "idade": int(idade), "curso": curso})
            st.success(f"{nome} cadastrado(a)!")
        else:
            st.error("Preencha nome e curso.")

# 2. Filtro
cursos = ["Todos"] + sorted(alunos.distinct("curso"))
escolha = st.selectbox("Filtrar por curso", cursos)
filtro = {} if escolha == "Todos" else {"curso": escolha}
dados = list(alunos.find(filtro, {"_id": 0, "nome": 1, "idade": 1, "curso": 1}).sort("nome", 1))

# 3. Indicadores e tabela
col1, col2 = st.columns(2)
col1.metric("Alunos", len(dados))
media = sum(d["idade"] for d in dados) / len(dados) if dados else 0
col2.metric("Idade média", f"{media:.1f}")

st.dataframe(pd.DataFrame(dados), hide_index=True)

Writing secretaria/app.py


---
## Parte 4 · Rodar o app no seu computador

O app roda pelo **terminal**, e não numa célula. Duas armadilhas comuns:

- O terminal do VS Code abre na pasta que você abriu no VS Code, que **nem sempre** é a pasta deste notebook. Por isso um simples `cd secretaria` pode dar *“não é possível localizar o caminho”*.
- O comando `python` do terminal pode ser **outro Python**, diferente do kernel deste notebook, e aí aparece *“No module named streamlit”*.

A célula abaixo resolve as duas: ela **escreve os comandos certos para o seu computador**, com o caminho completo da pasta e do Python.

In [8]:
import os, sys

pasta = os.path.abspath("secretaria")
print("Copie e cole no terminal (PowerShell), uma linha por vez:\n")
print(f'cd "{pasta}"')
print(f'& "{sys.executable}" -m streamlit run app.py')

Copie e cole no terminal (PowerShell), uma linha por vez:

cd "c:\Users\41228408807\Documents\GitHub\uniSENAI\Semestre_2\LABDA\NoSQL\Mongo\secretaria"
& "c:\Users\41228408807\AppData\Local\Programs\Python\Python314\python.exe" -m streamlit run app.py


1. No VS Code: **Terminal → New Terminal**.
2. Copie e cole no terminal as **duas linhas** que a célula acima mostrou (primeiro o `cd`, depois o `& ... streamlit run app.py`).
3. O navegador abre o app em `http://localhost:8501`.
4. Para parar: no terminal, **Ctrl+C**.

```
 navegador ──▶ app.py (Streamlit, no seu computador) ──▶ MongoDB (localhost:27017)
```

---
## Parte 5 · Conferir no banco

No app, **cadastre você mesmo** e teste o filtro por curso. Depois rode a célula abaixo: o seu cadastro, feito pelo site, aparece aqui no notebook, porque os dois usam o **mesmo banco**.

In [10]:
for a in alunos.find({}, {"_id": 0}).sort("nome", 1):
    print(a)
print(alunos.count_documents({}), "alunos")

{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação', 'disciplinas': ['Python']}
{'nome': 'Mohammad', 'idade': 20, 'curso': 'Dados'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
9 alunos


> Para colocar o app **na internet**, ele e o banco precisariam estar num servidor acessível de fora (por exemplo, Streamlit Community Cloud + MongoDB Atlas). Um banco em `localhost` só é visível no seu computador.

**Próximo:** `04_Banco_de_Imagens_GridFS.ipynb`